# Homework - Everything & Then Some

## Section 2 - Cleaning and pre-processing the data

In section 1 we only **looked** at the data. Now we start to **change** it, step by step:

| Step | What we do |
|---|---|
| 2.1 | Remove the nearly empty rows of the training set |
| 2.2 | Turn impossible values into missing values |
| 2.3 | Understand the missing values and decide how to fill them |
| 2.4 | Dates *(next)* |
| 2.5 | Categorical variables *(next)* |
| 2.6 | The optional `clientes.csv` table *(next)* |
| 2.7 | New features *(next)* |
| 2.8 | Scaling *(next)* |
| 2.9 | Everything in one reproducible `Pipeline` *(next)* |

### The golden rule of this section

> **Nothing learned from `test.csv` may be used to fit a transformation.**

To respect this rule we separate the operations into two kinds:

1. **Fixed rules** (e.g. "a negative number of days is impossible"). They do not *learn* anything from the data, so the same function can be applied to train and test.
2. **Fitted operations** (e.g. the median used to fill gaps, the scale of a variable, the list of categories). They *learn* numbers from the data, so they are fitted **only on the training set** and then applied to the test set. We will put these inside a scikit-learn `Pipeline` in step 2.9.

### 2.0 Libraries, configuration and loading the data

This is a new notebook, so we load the files again and recreate the lists of columns from section 1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)

# Random seed: makes the results reproducible
SEED = 42
np.random.seed(SEED)

# Name of the column we want to predict
TARGET = "repeat_purchase_90d"

# Folder with the CSV files (same as in the EDA notebook)
DATA_DIR = "../hdata"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
clientes = pd.read_csv(f"{DATA_DIR}/clientes.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print("train      :", train.shape)
print("test       :", test.shape)
print("clientes   :", clientes.shape)
print("sample_sub :", sample_sub.shape)

In [ ]:
# Same groups of columns as in section 1
id_cols = ["ID", "customer_id"]       # identifiers
date_col = "purchase_date"            # date (still text)
noise_col = "random_noise"            # pure noise (we confirmed it in the EDA)

cat_cols = ["payment_type", "salesperson", "commercial_zone",
            "transport_method", "warehouse", "document_series"]

num_cols = (train.select_dtypes(include="number")
                 .columns.difference([TARGET] + id_cols + [noise_col])
                 .tolist())

print(f"{len(num_cols)} numerical | {len(cat_cols)} categorical | {len(id_cols)} identifiers | 1 date | 1 noise")

### 2.1 Nearly empty rows

In the EDA we found **65 rows** of the training set where almost every column is missing, but the target is filled in.
Our decision is to **remove them from the training set**, for three reasons:

- with no information in the columns, there is nothing the model can learn from them;
- the **test set has no rows like this**, so we will never have to predict for a case like this;
- the target in those rows is balanced (about half 1s and half 0s), so removing them does not bias the target.

We use the same rule as in the EDA ("more than half of the informative columns are missing"), written as a function so that it is reproducible.

In [ ]:
# Informative columns: everything except the target and the identifiers
info_cols = [c for c in train.columns if c not in [TARGET] + id_cols]

def find_empty_rows(df, cols, max_missing_share=0.5):
    """True for the rows where more than `max_missing_share` of the columns are missing."""
    return df[cols].isna().mean(axis=1) > max_missing_share

empty_mask_train = find_empty_rows(train, info_cols)
empty_mask_test = find_empty_rows(test, info_cols)

print("Nearly empty rows in train:", int(empty_mask_train.sum()), "of", len(train))
print("Nearly empty rows in test :", int(empty_mask_test.sum()), "of", len(test))

print("\nTarget in the empty rows:")
print(train.loc[empty_mask_train, TARGET].value_counts().to_string())

**What we see:** 65 empty rows in the training set and **none** in the test set, as expected. Now we remove them from the training set.

Careful: we remove rows **only from train**. In the test set we can never remove rows, because we must predict **every** observation (the submission file needs all the `ID`s). That is why we also check that the test set has no empty rows: if it had, we would need another strategy.

In [ ]:
# The test set can never lose rows: we need a prediction for every ID
assert empty_mask_test.sum() == 0, "The test set has empty rows: we would need another strategy"

train_clean = train.loc[~empty_mask_train].reset_index(drop=True)

print("train before:", train.shape, "| after:", train_clean.shape)
print("Repurchase rate before:", round(train[TARGET].mean(), 3), "| after:", round(train_clean[TARGET].mean(), 3))

### 2.2 Impossible values (anomalies)

In the EDA we saw two kinds of impossible values:

- the value **1 000 000 000** in `prior_mean_purchase_value` and `spend_365d`: a placeholder, not a real amount;
- **negative values** in `days_since_previous_purchase` and `prior_mean_gap_days` (the number of days between two purchases cannot be negative) and in `prior_mean_purchase_value` and `spend_365d` (an average purchase value and a total spent in a year cannot be negative).

First, a quick look at the minimum and maximum of these four columns (so this notebook does not depend on the previous one).

In [ ]:
suspect_cols = ["days_since_previous_purchase", "prior_mean_gap_days",
                "prior_mean_purchase_value", "spend_365d"]

print(train_clean[suspect_cols].agg(["min", "max"]).T.to_string(float_format=lambda x: f"{x:,.2f}"))

**What we see:** the maxima are exactly 1 000 000 000 and all four columns have negative minima.

**Our decision:** convert these values to **missing values** (`NaN`). Why not the alternatives?

- *Delete the rows?* They are only about 1.5% of the training set, but the **test set also has some** and we cannot delete test rows.
- *Replace by a "reasonable" number?* That would be inventing data.
- *Keep them as they are?* A value of 1 000 000 000 would ruin averages and any scaling.

A `NaN` honestly says "we do not know this value", and it is handled together with the other missing values in step 2.3.

This is a **fixed rule** (it comes from common sense, not from learning on the data), so the same function is applied to train and test.

In [ ]:
PLACEHOLDER = 1_000_000_000
placeholder_cols = ["prior_mean_purchase_value", "spend_365d"]

def fix_anomalies(df):
    """Turns impossible values into NaN. Returns a copy; the original is not changed."""
    df = df.copy()
    for c in placeholder_cols:
        df.loc[df[c] >= PLACEHOLDER, c] = np.nan      # placeholder value
    for c in suspect_cols:
        df.loc[df[c] < 0, c] = np.nan                   # negative values are impossible here
    return df

# How many missing values each column had before the fix
na_before_train = train_clean[suspect_cols].isna().sum()
na_before_test = test[suspect_cols].isna().sum()

train_clean = fix_anomalies(train_clean)
test_clean = fix_anomalies(test)

summary = pd.DataFrame({
    "train_new_NaN": train_clean[suspect_cols].isna().sum() - na_before_train,
    "test_new_NaN":  test_clean[suspect_cols].isna().sum() - na_before_test,
})
print("Values turned into NaN:")
print(summary)

# Sanity checks: if one of these fails, the fix did not work
for df in [train_clean, test_clean]:
    assert (df[suspect_cols].dropna() >= 0).all().all()
    assert df[placeholder_cols].max().max() < PLACEHOLDER
print("\nSanity checks passed.")

**What we see:** the two "days" columns received 16 new `NaN` each in train. `prior_mean_purchase_value` received 32 and `spend_365d` 30, because they had both negative values and placeholders. In test, each column received 2 to 4. After the fix, the minima are no longer negative and the 1 000 000 000 disappeared:

In [ ]:
print(train_clean[suspect_cols].agg(["min", "max"]).T.to_string(float_format=lambda x: f"{x:,.2f}"))

### 2.3 Missing values

Now every missing value in the data is a `NaN`. Before filling anything, we need to understand **why** each one is missing, because the reason decides the treatment.

First, the overview: how many missing values per column, in train and in test?

In [ ]:
def missing_summary(a, b):
    """Missing values per column in two tables (here train and test), showing only the columns with any."""
    out = pd.DataFrame({
        "train_n": a.isna().sum(), "train_%": (a.isna().mean() * 100).round(1),
        "test_n":  b.isna().sum(), "test_%":  (b.isna().mean() * 100).round(1),
    })
    return out[(out["train_n"] > 0) | (out["test_n"] > 0)].sort_values("train_n", ascending=False)

print(missing_summary(train_clean.drop(columns=[TARGET]), test_clean))

**What we see:** 12 columns have missing values: seven `prior_*`-type columns, `spend_365d` and four categorical columns. Train and test have the same columns with gaps. The percentages of the `prior_*` columns are lower in test (we will come back to this). Let's separate the causes.

**Cause 1 - structural: the customer's first purchase.** If `prior_purchase_count` is 0, there is no history, so every `prior_*` statistic simply does not exist.

In [ ]:
prior_cols = ["days_since_previous_purchase", "prior_mean_gap_days", "prior_std_gap_days",
              "prior_min_purchase_value", "prior_max_purchase_value",
              "prior_mean_purchase_value", "prior_std_purchase_value"]

first_purchase = train_clean["prior_purchase_count"] == 0
print("First purchases in train:", int(first_purchase.sum()), "of", len(train_clean))

print("\n% missing among first purchases:")
print((train_clean.loc[first_purchase, prior_cols].isna().mean() * 100).round(1).to_string())

**What we see:** among the first purchases, **100%** of the seven columns are missing. This is a *structural* absence, as we suspected in the EDA.

(There is a smaller version of the same thing: the standard deviations need at least 2 or 3 earlier purchases, so they are also missing for customers with very little history.)

**Cause 2 - what is left?** If we look only at customers with **at least 3 earlier purchases**, every `prior_*` value should exist. Whatever is still missing there has some other origin. We check it together with `spend_365d` and the four categorical columns.

In [ ]:
cat_nan_cols = ["payment_type", "salesperson", "transport_method", "warehouse"]   # the categorical columns with NaN

enough_history = train_clean["prior_purchase_count"] >= 3
check_cols = prior_cols + ["spend_365d"] + cat_nan_cols

print("% missing among purchases with at least 3 earlier purchases:")
print((train_clean.loc[enough_history, check_cols].isna().mean() * 100).round(1).to_string())

**What we see:**
- Four columns (`prior_std_gap_days`, `prior_min_purchase_value`, `prior_max_purchase_value`, `prior_std_purchase_value`) are **never** missing here, so their gaps are **purely structural**.
- All the others (`days_since_previous_purchase`, `prior_mean_gap_days`, `prior_mean_purchase_value`, `spend_365d` and the four categorical columns) are still missing in **about 3%** of the rows, with similar percentages. A small part of this (about 0.25% to 0.5%) comes from the anomalies we turned into `NaN` in step 2.2; the rest looks like gaps scattered at random, not a structural cause.

Do these random gaps come in blocks (the same rows missing several columns at once), or are they independent? For the four categorical columns:

In [ ]:
n_missing_cat = train_clean[cat_nan_cols].isna().sum(axis=1)
print("Rows by number of missing categorical columns (out of 4):")
print(n_missing_cat.value_counts().sort_index().to_string())

**What we see:** most rows have no gap, about 11% have exactly one, and very few have two or more (about 0.5%). That is what we would expect if each column lost values **independently** (6 pairs of columns × 3% × 3% ≈ 0.5%). So they are not blocks of bad rows: they are isolated random gaps.

**Is the absence itself informative?** If the target behaves differently when a value is missing, the fact that it is missing is a useful signal. We compare the repurchase rate when the value is missing vs present.

In [ ]:
rows = []
for c in cat_nan_cols + ["spend_365d"]:
    m = train_clean[c].isna()
    rows.append({"column": c,
                 "n_missing": int(m.sum()),
                 "rate_if_missing": round(train_clean.loc[m, TARGET].mean(), 3),
                 "rate_if_present": round(train_clean.loc[~m, TARGET].mean(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

print("\nRepurchase rate: first purchase =", round(train_clean.loc[first_purchase, TARGET].mean(), 3),
      "| not first purchase =", round(train_clean.loc[~first_purchase, TARGET].mean(), 3))

**What we see:**
- For the random gaps, the two rates are mostly within a few points of each other and of the overall rate (about 56%). The biggest difference is `warehouse` (about 64% vs 56%), but with only about 195 rows that is only about 2 standard errors, so it is borderline and not worth a special rule. Giving the categorical gaps their own "missing" category (below) will let the model use that signal anyway, if it exists.
- For **first purchases**, the repurchase rate is much lower (about 33% vs 58%). Being a first purchase **is informative**, and since that information would be lost if we only filled the gaps, we will keep it as an explicit feature.

One more observation, which is not a cleaning decision but will matter later:

In [ ]:
first_purchase_test = test_clean["prior_purchase_count"] == 0
print("Share of first purchases | train:", round(first_purchase.mean() * 100, 1), "%",
      "| test:", round(first_purchase_test.mean() * 100, 1), "%")

**What we see:** first purchases are much rarer in the test set than in the training set. This is natural: the test is in the future, so customers have accumulated more history. It is a first sign that **the test set does not look exactly like the training set**, and one more reason to use a temporal validation in section 4.

#### Decisions for the missing values

| Type of missing | Columns | Decision |
|---|---|---|
| Structural (first purchase) | the seven `prior_*`-type columns | create the flag `is_first_purchase`, **and** fill the gaps with the median |
| Structural (little history) | `prior_std_gap_days`, `prior_std_purchase_value` (they need 2 or 3 earlier purchases) | fill with the median |
| Random, about 3% | `days_since_previous_purchase`, `prior_mean_gap_days`, `prior_mean_purchase_value`, `spend_365d` | fill with the **median** |
| Random, about 3% | `payment_type`, `salesperson`, `transport_method`, `warehouse` | new category **"missing"** |

Why these choices?
- **Median, not mean:** the variables are very skewed with a few very large values, and the mean would be pulled up by them.
- **Category "missing", not the most frequent one:** with a variable such as `payment_type`, where one category has 95% repurchase, guessing the most frequent category would put wrong information into the rows. An honest "missing" label is safer.
- **Flag + median for the first purchases:** the median fills the hole so that models that cannot handle `NaN` work, and the flag tells the model that the value was not real.

**Important:** the median is a number *learned from the data*, so it is **fitted only on the training set** (inside the `Pipeline`, step 2.9) and then applied to test. The flag `is_first_purchase` is a fixed rule, so we can create it right now for both tables.

In [ ]:
def add_first_purchase_flag(df):
    """Fixed rule: 1 if the customer has no earlier purchase, 0 otherwise."""
    df = df.copy()
    df["is_first_purchase"] = (df["prior_purchase_count"] == 0).astype(int)
    return df

train_clean = add_first_purchase_flag(train_clean)
test_clean = add_first_purchase_flag(test_clean)

print(train_clean["is_first_purchase"].value_counts().to_string())
print("\ntrain_clean:", train_clean.shape, "| test_clean:", test_clean.shape)

---
### Summary of steps 2.1 to 2.3

| Step | Decision | Fixed rule or fitted? |
|---|---|---|
| 2.1 | Remove the 65 nearly empty rows (train only; the test has none) | Fixed rule |
| 2.2 | 1 000 000 000 and negative values (the impossible ones) become `NaN` | Fixed rule |
| 2.3 | Flag `is_first_purchase`; numerical gaps filled with the median; categorical gaps become "missing" | Flag: fixed rule. Median: **fitted on train** (step 2.9) |

At this point we have `train_clean` and `test_clean`, with the same columns (plus the target in `train_clean`).

**Next:** 2.4 dates, 2.5 categorical variables, 2.6 `clientes.csv`, 2.7 new features, 2.8 scaling, 2.9 the `Pipeline`.